# MMS–bow-shock magnetic connection

This notebook mirrors `mms_bow_shock_connection.py`: it extracts a bow shock from a BATSRUS Tecplot file, derives a symmetric MMS interval from the Tecplot event time, loads that interval in GSM, uses the averaged `Bavg` direction as a straight field line, finds the closest shock intersection, and displays 2D and 3D views.

Install the optional MMS/Jupyter dependencies with `pip install -e ".[mms,notebook]"`. Launch it from the repository root with:

```bash
jupyter lab examples/shock_connection.ipynb
```

The Tecplot sample is large and the MMS cells download data from the network. Keep the notebook unexecuted when committing it.

## Parameters

Edit the input path, MMS window duration, extraction resolution, and smoothing controls before running the analysis. The window is centered automatically on the Tecplot event timestamp, and the MMS products are requested in GSM so their position and magnetic field share the Tecplot coordinate system.

In [ ]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import pyvista as pv

from shocklink.bowshock import (
    calc_bow_shock_normals,
    extract_shockfit_range,
    fit_bow_shock,
    get_bow_shock_surface,
    smooth_bow_shock_surface,
)
from shocklink.connectivity import (
    analyze_shock_connection,
    plot_shock_angle_contour,
    plot_shock_connection_3d,
)
from shocklink.dataset import calc_velocity_divergence
from shocklink.mms import average_plotted_values, load_mms_data
from shocklink.io import TIME_EVENT_KEY, load_simulation
from shocklink.utilities import parse_datetime

pv.set_jupyter_backend("static")

DATA_PATH = Path("../data/3d.dat")
MMS_WINDOW_SECONDS = 300.0
PROBE = 1
MODE = "auto"
SURFACE_AXIS = np.linspace(-30.0, 30.0, 241)
X_RESOLUTION = 512
CHUNK_SIZE = 1024
SMOOTHING_SIGMA = 5.0
SHOCKFIT_RANGE = (-5.0, 5.0)

## Extract and smooth the bow shock

The extracted `surface_x` array is indexed as `[Y, Z]`. Missing columns remain `NaN`; the normal calculation fills supported gaps for differentiation, while the connection plots keep unsupported shock cells masked.

In [ ]:
grid = load_simulation(DATA_PATH)
event_text = str(np.asarray(grid.field_data[TIME_EVENT_KEY]).reshape(-1)[0])
event_time = parse_datetime(event_text)
mms_start = event_time - timedelta(seconds=MMS_WINDOW_SECONDS / 2.0)
mms_end = event_time + timedelta(seconds=MMS_WINDOW_SECONDS / 2.0)
MMS_START = mms_start.isoformat()
MMS_END = mms_end.isoformat()
calc_velocity_divergence(grid)
fit = fit_bow_shock(grid)
shock_region = extract_shockfit_range(
    grid,
    lower=SHOCKFIT_RANGE[0],
    upper=SHOCKFIT_RANGE[1],
)

surface_x_raw = get_bow_shock_surface(
    shock_region,
    x_resolution=X_RESOLUTION,
    chunk_size=CHUNK_SIZE,
    refine_minimum=True,
)
surface_x = smooth_bow_shock_surface(surface_x_raw, sigma=SMOOTHING_SIGMA)
normals = calc_bow_shock_normals(
    surface_x, y=SURFACE_AXIS, z=SURFACE_AXIS
)

print(f"Tecplot event: {grid.field_data[TIME_EVENT_KEY]}")
print(f"Fitted nose X: {fit.loc0[0]:.3f} R_E")
print(f"Finite surface samples: {np.isfinite(surface_x).sum():,}/{surface_x.size:,}")

## Load MMS and calculate the connection

MMS averages are resolved in GSM over the symmetric window derived from the Tecplot event. The normalized `Bavg` vector defines an infinite straight line in both directions; if it crosses the shock more than once, the result selects the crossing closest to MMS.

In [ ]:
mms_data = load_mms_data(
    MMS_START,
    MMS_END,
    probe=PROBE,
    mode=MODE,
    coordinates="gsm",
)
averages = average_plotted_values(mms_data)
mms_position = np.array(
    [averages[f"satellite_location_{axis}"] for axis in "xyz"],
    dtype=float,
)
bavg = np.array(
    [averages[f"magnetic_field_{axis}"] for axis in "xyz"],
    dtype=float,
)

connection = analyze_shock_connection(
    surface_x,
    normals,
    y=SURFACE_AXIS,
    z=SURFACE_AXIS,
    mms_position=mms_position,
    bavg=bavg,
)
hit = connection.selected_intersection
print(f"MMS interval: {MMS_START} to {MMS_END} (MMS{PROBE}, GSM)")
print(f"MMS position [R_E]: {mms_position}")
print(f"Bavg [nT]: {bavg}")
print(f"Intersection [R_E]: {hit.point}")
print(f"Signed line parameter: {hit.line_parameter:.6g}")
print(f"Distance from MMS: {hit.distance:.6g} R_E")
print(f"theta_Bn: {hit.theta_bn_deg:.3f} deg")

## Plot the connection

The 2D view shows the acute `theta_Bn` field over the extracted Y–Z shock and marks the selected intersection. The 3D view adds Earth, the colored shock, MMS, the straight field line, the Bavg arrow, and the intersection.

In [ ]:
figure, axes = plot_shock_angle_contour(connection)

In [ ]:
plotter = plot_shock_connection_3d(connection, show=False)
plotter.show(jupyter_backend="static")